# Introduction
This notebook will contain different API-calling tests for our ATS systems.

In [2]:
# setup
import requests
import pandas as pd
from collections import defaultdict
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import time
from functools import reduce

session = requests.Session()
retries = Retry(total=3, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504])
session.mount('https://', HTTPAdapter(max_retries=retries))

# Greenhouse.io

In [3]:
# import greenhouse company list
greenhouse = pd.read_csv('../ats_companies/greenhouse.csv')
greenhouse.head()

,slug
0,0verwatch
1,103644278
2,10alabs
3,10beauty
4,10xgenomics


In [52]:
# define fetching scheme
def get_greenhouse(slug: str) -> dict:
    try:
        res = session.get(f'https://boards-api.greenhouse.io/v1/boards/{slug}/jobs')
        if res.ok:
            res_json = res.json()
            if res_json['meta']['total'] > 0:
                return res_json['jobs']
            else:
                return []
        elif res.status_code == 404:
            raise Exception('404 - job not found')
        else:
            raise Exception(res.status_code, '- error')
    except Exception as e:
        return str(e)

def scrape_greenhouse(
        df: pd.DataFrame, 
        limit: int= -1
    ) -> dict:
    '''Scrape greenhouse.io for jobs using a pd.DataFrame containing company slugs.'''
    # greenhouse.io defines no rate limit on their job board API, however they define a 50-per-10-second
    # limit on their audit log API, so we will respect this.
    # 5 per second, .2s delay
    # this takes about 50 minutes

    # --- create variables
    results = defaultdict()
    i = 0
    # stats
    valid = 0
    invalid = 0
    has_jobs = 0
    total_jobs = 0

    # --- begin scrape
    start_time = time.time()
    for slug in df.slug:
        if i == limit: break
        # - fetch
        res = get_greenhouse(slug)

        # - validate
        if isinstance(res, list):
            valid += 1
            if len(res) > 0:
                has_jobs += 1
                total_jobs += len(res)
            print(f'{i:<4} {slug:<40}           valid response              total: {len(res)}')
        else:
            invalid += 1
            print(f'{i:<4} {slug:<40}           invalid response            {res}')

        # - update
        results[slug] = res
        i += 1

        # respect rate limit
        time.sleep(0.2)
    end_time = time.time()

    # --- print stats
    print('-'*103)
    if i > 0:
        print(f'''scrape results:
    scraping time:        {end_time-start_time:.1f}s ({(end_time-start_time) / 60:.2f}m)
    scraped slugs:        {i}
    total jobs:           {total_jobs}
    -
    valid slugs:          {valid} ({(valid/i)*100:.1f}%)
    valid with jobs:      {has_jobs} 
    invalid:              {invalid} ({(invalid/i)*100:.1f}%)
''')
    else:
        print('no jobs scraped.')

    return results

def process_greenhouse_results(
        results: dict, 
        filepath: str='./greenhouse_jobs.csv'
    ) -> pd.DataFrame:

    df = pd.DataFrame(
        columns=[
            'slug', 'absolute_url', 'internal_job_id', 
            'company_name', 'requisition_id', 
            'title', 'first_published', 'language'],
    )

    # iterate through all slugs, and extract information from all valid job listings
    for slug in results.keys():
        jobs = results[slug]
        # process jobs
        if isinstance(jobs, list) and len(jobs) > 0:
            rows = []
            for job in jobs:
                data = {
                    'slug': slug, 
                    'absolute_url':     job['absolute_url'], 
                    'internal_job_id':  job['internal_job_id'], 
                    'company_name':     str(job['company_name']).strip(), 
                    'requisition_id':   job['requisition_id'], 
                    'title':            str(job['title']).strip().lower(), 
                    'first_published':  job['first_published'], 
                    'language':         str(job['language']).strip().lower(), 
                }
                rows.append(data)
            rows_df = pd.DataFrame(rows)
            df = pd.concat([df, rows_df], ignore_index=True)

    # add timestamp
    df['scrape_timestamp'] = pd.Timestamp.now(tz='UTC')
    # save as csv
    df.to_csv(filepath)
    return df

def process_greenhouse(
        df: pd.DataFrame,
        limit: int=-1,
        filepath: str='./greenhouse_jobs.csv'
    ) -> pd.DataFrame:
    results = scrape_greenhouse(df, limit)
    df = process_greenhouse_results(results, filepath)
    return df

In [54]:
process_greenhouse(greenhouse)

0    0verwatch                                          valid response              total: 1
1    103644278                                          valid response              total: 16
2    10alabs                                            valid response              total: 16
3    10beauty                                           valid response              total: 8
4    10xgenomics                                        invalid response            404 - job not found
5    12jlkfsk                                           valid response              total: 5
6    12twenty                                           valid response              total: 3
7    143studiosinc                                      valid response              total: 4
8    1456754456yhgbhfg                                  valid response              total: 1
9    14technology                                       valid response              total: 2
10   1800contacts                                       v

,slug,absolute_url,internal_job_id,company_name,requisition_id,title,first_published,language,scrape_timestamp
0,0verwatch,https://job-boards.greenhouse.io/0verwatch/job...,4175494009,0verwatch,11,full stack engineer,2026-06-26T14:54:10-04:00,en,2026-09-28 23:24:06.225181+00:00
1,103644278,https://job-boards.greenhouse.io/103644278/job...,3529091,Lume Deodorant,1977,"associate creative director, art director",2026-08-24T16:50:22-04:00,en,2026-09-28 23:24:06.225181+00:00
2,103644278,https://job-boards.greenhouse.io/103644278/job...,3488538,Lume Deodorant,1953,"associate creative director, copywriter",2026-07-07T12:37:12-04:00,en,2026-09-28 23:24:06.225181+00:00
3,103644278,https://job-boards.greenhouse.io/103644278/job...,3505581,Lume Deodorant,1964,head of tiktok shop,2026-07-28T12:32:03-04:00,en,2026-09-28 23:24:06.225181+00:00
4,103644278,https://job-boards.greenhouse.io/103644278/job...,3549548,Lume Deodorant,1984,"influencer marketing manager, lume and mando",2026-09-14T13:50:50-04:00,en,2026-09-28 23:24:06.225181+00:00
...,...,...,...,...,...,...,...,...,...
151018,zyngacareers,https://job-boards.greenhouse.io/zyngacareers/...,5215617004,Zynga,R_118688,user acquisition manager,2026-09-16T02:20:02-04:00,en,2026-09-28 23:24:06.225181+00:00
151019,zyngacareers,https://job-boards.greenhouse.io/zyngacareers/...,5219238004,Zynga,R_118727,vip account manager,2026-09-22T11:13:58-04:00,en,2026-09-28 23:24:06.225181+00:00
151020,zyngaearlycareers,https://job-boards.greenhouse.io/zyngaearlycar...,5215618004,Zynga Early Careers,R_118698,marketing artist intern,2026-09-17T11:50:31-04:00,en,2026-09-28 23:24:06.225181+00:00
151021,zyngaearlycareers,https://job-boards.greenhouse.io/zyngaearlycar...,5214860004,Zynga Early Careers,R_118724,rotational product manager - canada,2026-09-15T13:26:47-04:00,en,2026-09-28 23:24:06.225181+00:00


In [56]:
sm = pd.read_csv('../ats_companies/smartrecruiters.csv')
sm.head()

,slug
0,10xValuePartnersGmbH
1,1839Labs
2,1HUDDLE
3,1Huddle
4,1perTInentInc


In [64]:
res = requests.get('https://api.smartrecruiters.com/v1/companies/1huddle/postings')

In [65]:
res.json()['content']

[{'id': '744000056465632',
  'name': 'Associate, Internship Team',
  'uuid': 'b4bf7a54-c438-4e39-aedd-32f156894b4e',
  'jobAdId': '353f4b98-0e8c-4c46-ab92-8dac6a53e5cf',
  'defaultJobAd': True,
  'refNumber': 'REF7R',
  'company': {'identifier': '1Huddle', 'name': '1Huddle'},
  'releasedDate': '2025-04-29T17:58:57.823Z',
  'location': {'city': 'Newark',
   'region': 'NJ',
   'country': 'us',
   'address': '550 Broad Street',
   'postalCode': '07102',
   'remote': False,
   'hybrid': False,
   'latitude': '40.7425757',
   'longitude': '-74.1687227',
   'fullLocation': 'Newark, NJ, United States'},
  'industry': {'id': 'computer_software', 'label': 'Computer Software'},
  'department': {},
  'function': {'id': 'business_development', 'label': 'Business Development'},
  'typeOfEmployment': {'id': 'intern', 'label': 'Intern'},
  'experienceLevel': {'id': 'internship', 'label': 'Internship'},
  'customField': [{'fieldId': 'COUNTRY',
    'fieldLabel': 'Country/Region',
    'valueId': 'us',
 

In [62]:
res2 = requests.get('https://api.smartrecruiters.com/v1/companies/1HUDDLE/postings/744000056465632')

In [63]:
res2.json()

{'id': '744000056465632',
 'name': 'Associate, Internship Team',
 'uuid': 'b4bf7a54-c438-4e39-aedd-32f156894b4e',
 'jobId': '4fd2fe3a-f3bd-4798-b4ef-75661bd2c6d6',
 'jobAdId': '353f4b98-0e8c-4c46-ab92-8dac6a53e5cf',
 'defaultJobAd': True,
 'refNumber': 'REF7R',
 'company': {'name': '1Huddle', 'identifier': '1Huddle'},
 'location': {'city': 'Newark',
  'region': 'NJ',
  'country': 'us',
  'address': '550 Broad Street',
  'postalCode': '07102',
  'remote': False,
  'hybrid': False,
  'latitude': '40.7425757',
  'longitude': '-74.1687227',
  'fullLocation': 'Newark, NJ, United States'},
 'customField': [{'fieldId': 'COUNTRY',
   'fieldLabel': 'Country/Region',
   'valueId': 'us',
   'valueLabel': 'United States'},
  {'fieldId': '58b7e892e4b09a6d37a0d5df',
   'fieldLabel': 'Brands',
   'valueId': 'default',
   'valueLabel': '1Huddle'}],
 'releasedDate': '2025-04-29T17:58:57.823Z',
 'creator': {'name': 'Brianna Hernandez',
  'avatarUrl': 'https://c.smartrecruiters.com/sr-employee-image-prod